<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB14_Sequence_Models_RNN_LSTM_Real_Fuel_Forecasting_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB14 · Clase 14 — Modelos de secuencias: RNN, LSTM y previsión real de combustible**

## Bloque 3: IA — Deep Learning (continuación)

`NB11`–`NB13` trabajaron con datos tabulares e imágenes. Esta clase cubre la tercera gran forma de dato: las **secuencias**, donde el significado de un valor depende de lo que vino antes. Construimos la teoría desde cero — la relación de recurrencia de la RNN, el problema del desvanecimiento del gradiente, y cómo lo soluciona LSTM — y después la aplicamos de verdad: replanteando los datos de consumo de combustible de buques de `NB07`/`NB09` como lo que realmente son, una **serie temporal por buque**, y entrenando una LSTM para prever el consumo de combustible del mes siguiente a partir de los meses anteriores.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar por qué los datos secuenciales necesitan una arquitectura distinta a un MLP o una CNN.
- Explicar la relación de recurrencia de la RNN y calcular a mano los estados ocultos de una RNN pequeña.
- Explicar el problema del desvanecimiento del gradiente y cómo lo aborda la celda de estado con puertas de la LSTM.
- Remodelar datos tabulares reales con un eje temporal natural en secuencias adecuadas para un modelo recurrente, sin filtrar información entre buques.
- Entrenar y evaluar una LSTM real de PyTorch, juzgada frente a una base de referencia simple y honesta.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, hoja de ruta de hoy | 5 min | Teoría |
| 2 | ¿Por qué modelos de secuencias? Dónde fallan los MLP y las CNN | 10 min | Teoría |
| 3 | La relación de recurrencia de la RNN, desplegada en el tiempo | 15 min | Teoría + Práctica |
| 4 | Ejemplo resuelto: calcular a mano los estados ocultos de una RNN | 15 min | Teoría + Práctica |
| 5 | El problema del desvanecimiento del gradiente | 10 min | Teoría + Práctica |
| 6 | LSTM: puertas y estado de celda | 15 min | Teoría |
| 7 | Dataset real: los datos de combustible de buques como serie temporal propiamente dicha | 10 min | Práctica |
| 8 | Manos a la obra: construir secuencias y entrenar un pronosticador LSTM | 20 min | Práctica |
| 9 | Evaluar la LSTM frente a una base de referencia honesta | 15 min | Práctica |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son una orientación aproximada, no un guion estricto — no hay descansos programados. Si cubrimos todo con tiempo de sobra, la clase termina antes; puede pasar y está bien.

---

## 1. Repaso: dónde estamos

- **`NB11`**: perceptrón → MLP, un primer clasificador real de PyTorch.
- **`NB12`**: entrenar redes profundas correctamente — monitorización de validación, dropout, parada temprana, optimizadores.
- **`NB13`**: redes neuronales convolucionales sobre imágenes reales de inspección submarina de casco.
- **`NB14`** (hoy): modelos de secuencias — RNN, LSTM — aplicados a una serie temporal naval real.

---

## 2. ¿Por qué modelos de secuencias? Dónde fallan los MLP y las CNN

El MLP de `NB11` trataba sus entradas como una bolsa desordenada de números. La CNN de `NB13` respeta la estructura *espacial*, pero no el *orden temporal*. Algunos datos navales/oceánicos reales son fundamentalmente **secuenciales**: el significado de un valor depende de lo que vino antes.

Recupera `ship_fuel_efficiency.csv` de `NB07`/`NB09`: cada uno de los 120 buques tiene **12 registros mensuales**, de enero a diciembre. Todos los modelos que hemos entrenado con él hasta ahora — regresión logística, Random Forest, K-Means — trataron esas 1.440 filas como ejemplos independientes e intercambiables, exactamente igual que el planteamiento de clasificación/regresión de `NB07` y el clustering de `NB09`. Eso descarta algo real: `el consumo de combustible de un buque este mes no es independiente de su consumo en los meses recientes` — las rutas, los ciclos de mantenimiento y los patrones estacionales del tiempo se arrastran de un mes a otro. Las **redes neuronales recurrentes (RNN)** están construidas específicamente para llevar información hacia delante de un paso de la secuencia al siguiente, en vez de tratar cada fila como un sorteo nuevo y sin relación con los demás.

---

## 3. La relación de recurrencia de la RNN, desplegada en el tiempo

Una **celda RNN** toma la entrada actual $x_t$ *y* su propia salida anterior — el **estado oculto** $h_{t-1}$ — y produce un nuevo estado oculto:

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
$$

$W_x$, $W_h$ y $b$ son **los mismos pesos en cada paso temporal** — una única celda pequeña, aplicada repetidamente, que lleva $h_t$ hacia delante como memoria de todo lo visto hasta ese momento. "Desplegar" dibuja esa misma celda una vez por cada paso temporal, para ver la secuencia completa de un vistazo:

Dibujemos esa estructura desplegada para una secuencia de 4 pasos:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(11, 4.5))

n_steps = 4
box_w, box_h = 1.2, 1.0
xs = [i * 2.6 for i in range(n_steps)]

for i, x in enumerate(xs):
    ax.add_patch(patches.FancyBboxPatch((x, 0), box_w, box_h, boxstyle="round,pad=0.05",
                                         facecolor="lightcoral", edgecolor="black"))
    ax.text(x + box_w / 2, box_h / 2, "RNN\ncell", ha="center", va="center", fontsize=9)

    ax.annotate("", xy=(x + box_w / 2, 0), xytext=(x + box_w / 2, -1),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text(x + box_w / 2, -1.3, f"$x_{i + 1}$", ha="center", fontsize=11)

    ax.annotate("", xy=(x + box_w / 2, box_h + 1), xytext=(x + box_w / 2, box_h),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text(x + box_w / 2, box_h + 1.3, f"$y_{i + 1}$", ha="center", fontsize=11)

    if i < n_steps - 1:
        ax.annotate("", xy=(xs[i + 1], box_h / 2), xytext=(x + box_w, box_h / 2),
                    arrowprops=dict(arrowstyle="->", lw=2, color="darkblue"))
        ax.text(x + box_w + (xs[i + 1] - x - box_w) / 2, box_h / 2 + 0.3,
                f"$h_{i + 1}$", ha="center", color="darkblue", fontsize=10)

ax.annotate("", xy=(xs[0], box_h / 2), xytext=(xs[0] - 0.8, box_h / 2),
            arrowprops=dict(arrowstyle="->", lw=2, color="darkblue"))
ax.text(xs[0] - 1.0, box_h / 2 + 0.3, "$h_0$", ha="center", color="darkblue", fontsize=10)

ax.text(xs[-1] / 2, -2.0, "Same cell, same weights, reused at every time step",
        ha="center", fontsize=10, style="italic")

ax.set_xlim(-2, xs[-1] + box_w + 1)
ax.set_ylim(-2.5, box_h + 2)
ax.axis("off")
ax.set_title("An RNN unrolled through time")
plt.tight_layout()
plt.show()

Esto es una única celda RNN dibujada cuatro veces, no cuatro celdas distintas — las flechas azules que llevan $h_t$ hacia delante son la forma en que `la información de principios de una secuencia (por ejemplo, la eficiencia de combustible de enero) puede influir en una predicción mucho más adelante` (por ejemplo, una previsión para diciembre).

> **Para saber más**: [Red neuronal recurrente (Wikipedia)](https://en.wikipedia.org/wiki/Recurrent_neural_network).

---

## 4. Ejemplo resuelto: calcular a mano los estados ocultos de una RNN

Números pequeños, sin entrenar, puramente ilustrativos — el mismo espíritu que la convolución calculada a mano en `NB13` — para que la fórmula de recurrencia deje de ser abstracta:

In [ ]:
import numpy as np

def rnn_cell(x, h_prev, Wx, Wh, b):
    return np.tanh(Wx * x + Wh * h_prev + b)

# Toy weights -- not learned, just for illustration
Wx, Wh, b = 0.5, 0.8, 0.1

x_sequence = [1.0, 0.5, -0.5, 1.0]
h = 0.0  # initial hidden state h0

print(f"h0 = {h:.3f}")
for t, x_t in enumerate(x_sequence, start=1):
    h = rnn_cell(x_t, h, Wx, Wh, b)
    print(f"x{t} = {x_t:>5} -> h{t} = {h:.3f}")

**Sigue el rastro tú mismo**: cada $h_t$ depende tanto de $x_t$ *como* de todos los $h$ anteriores, a través de la cadena de sustituciones — $h_4$ está influido por $x_1$, tres pasos antes, enteramente a través de $h_1 \to h_2 \to h_3$.

**Pruébalo tú mismo**: cambia los pesos de juguete e introduce un único "pulso" — una entrada no nula al principio, y luego ceros — para ver directamente el efecto de memoria: ¿cuánto sigue influyendo esa primera entrada en $h_4$, tres pasos después, únicamente a través de $W_h$?

In [ ]:
Wx2, Wh2, b2 = 0.2, 1.2, -0.1
x_sequence2 = [1.0, 0.0, 0.0, 0.0]   # a single pulse at t=1, then nothing

h2 = 0.0
print(f"h0 = {h2:.3f}")
for t, x_t in enumerate(x_sequence2, start=1):
    h2 = rnn_cell(x_t, h2, Wx2, Wh2, b2)
    print(f"x{t} = {x_t:>5} -> h{t} = {h2:.3f}")


---

## 5. El problema del desvanecimiento del gradiente

Entrenar una RNN usa **retropropagación a través del tiempo** — la regla de la cadena estirada hacia atrás a lo largo de cada paso temporal. Como el *mismo* peso $W_h$ multiplica el estado oculto en cada paso, el gradiente que fluye hacia atrás a lo largo de $n$ pasos involucra ese peso (y la derivada de $\tanh$, siempre $\le 1$) multiplicado por sí mismo aproximadamente $n$ veces. Si ese factor por paso está aunque sea ligeramente por debajo de 1, la multiplicación repetida lo reduce **exponencialmente** — ilustrado abajo con aritmética sencilla, no con un modelo realmente entrenado:

In [ ]:
factor = 0.6  # a representative per-step gradient factor below 1
steps = np.arange(0, 13)
magnitude = factor ** steps

plt.figure(figsize=(7, 4))
plt.plot(steps, magnitude, marker="o")
plt.yscale("log")
plt.xlabel("Number of time steps back-propagated through")
plt.ylabel("Relative gradient magnitude (log scale)")
plt.title(f"Illustrative effect of repeated multiplication by {factor} per step")
plt.show()

A los 10-12 pasos hacia atrás, este gradiente ilustrativo ya es órdenes de magnitud más pequeño que en el paso 1 — `en una RNN simple, la información de tan atrás apenas influye en el aprendizaje`. Nuestras secuencias de hoy tienen solo 11 pasos, justo en el rango donde esto empieza a notarse — una buena razón práctica para recurrir a LSTM en vez de a una RNN simple.

> **Para saber más**: [Problema del desvanecimiento del gradiente (Wikipedia)](https://en.wikipedia.org/wiki/Vanishing_gradient_problem).

---

## 6. LSTM: puertas y estado de celda

La **Long Short-Term Memory (LSTM)** añade una segunda vía — el **estado de celda** $C_t$ — que se transporta entre pasos temporales sobre todo mediante *suma* en vez de multiplicación repetida, controlada por tres puertas aprendidas:

| Puerta | Pregunta que responde |
|---|---|
| **Puerta de olvido** $f_t$ | ¿Cuánto del antiguo estado de celda deberíamos conservar? |
| **Puerta de entrada** $i_t$ | ¿Cuánta información nueva deberíamos añadir? |
| **Puerta de salida** $o_t$ | ¿Cuánto del estado de celda debería influir en la salida de este paso? |

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \qquad h_t = o_t \odot \tanh(C_t)
$$

Como la actualización de $C_t$ está dominada por la suma, y no por una reducción multiplicativa repetida, `las LSTM manejan dependencias mucho más largas que las RNN simples antes de que el problema del desvanecimiento del gradiente de la Parte 5 se imponga`. La más sencilla **[GRU](https://en.wikipedia.org/wiki/Gated_recurrent_unit)** (Gated Recurrent Unit) fusiona el estado de celda y el estado oculto y usa solo dos puertas — menos parámetros, a menudo competitiva en secuencias cortas como la nuestra; cambiar `nn.LSTM` por `nn.GRU` más adelante es un cambio de una línea, merece la pena probarlo como tarea.

> **Para saber más**: [Long short-term memory (Wikipedia)](https://en.wikipedia.org/wiki/Long_short-term_memory).

---

## 7. Dataset real: los datos de combustible de buques como serie temporal propiamente dicha

Vuelve a cargar el mismo `ship_fuel_efficiency.csv` real de `NB07`/`NB09` — 120 buques, cada uno con 12 registros mensuales cronológicos:

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
print(fuel.shape)
fuel[fuel["ship_id"] == fuel["ship_id"].iloc[0]][["ship_id", "month", "distance", "fuel_consumption", "engine_efficiency"]]

**Tarea de hoy**: usando los primeros **11 meses** de cada buque (`distance`, `fuel_consumption`, `engine_efficiency` — el mismo conjunto de características libre de fugas del clustering de `NB09`, excluyendo también `CO2_emissions`) como secuencia de entrada, predice el `fuel_consumption` del **mes 12** de ese buque. Construye una secuencia (11 pasos, 3 características) por buque, más su objetivo:

In [ ]:
import numpy as np

feature_cols = ["distance", "fuel_consumption", "engine_efficiency"]
ship_ids = fuel["ship_id"].unique()

sequences, targets = [], []
for sid in ship_ids:
    ship_rows = fuel[fuel["ship_id"] == sid][feature_cols].values  # already chronological, Jan-Dec
    sequences.append(ship_rows[:11])          # months 1-11 as input
    targets.append(ship_rows[11, 1])          # month 12's fuel_consumption (column index 1)

X_seq = np.stack(sequences)   # (120 ships, 11 months, 3 features)
y_seq = np.array(targets)     # (120,)
print(X_seq.shape, y_seq.shape)

Divide **por buque**, no por fila — meter distintos meses del *mismo* buque en train y en test filtraría información sobre el comportamiento global de combustible de ese buque a través de la división, repitiendo la lección de fuga de datos de `NB07` en una forma nueva, específica de secuencias:

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(ship_ids))
idx_train_full, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_val = train_test_split(idx_train_full, test_size=0.25, random_state=42)

print(f"Train ships: {len(idx_train)}  Val ships: {len(idx_val)}  Test ships: {len(idx_test)}")

**Pruébalo tú mismo**: confirma directamente la promesa de dividir por buque — comprueba que ningún `ship_id` aparece en más de una de las tres particiones.

In [ ]:
train_ships = set(ship_ids[idx_train])
val_ships = set(ship_ids[idx_val])
test_ships = set(ship_ids[idx_test])

print("Train/val overlap:", train_ships & val_ships)
print("Train/test overlap:", train_ships & test_ships)
print("Val/test overlap:", val_ships & test_ships)
print("All three splits together cover every ship?", len(train_ships | val_ships | test_ships) == len(ship_ids))


---

## 8. Manos a la obra: construir secuencias y entrenar un pronosticador LSTM

Escala las características usando estadísticas solo de las **secuencias de entrenamiento** (ajustado sobre los datos de train aplanados, exactamente igual que el escalado libre de fugas de todos los notebooks anteriores), y después conviértelas en tensores:

In [ ]:
from sklearn.preprocessing import StandardScaler
import torch

scaler = StandardScaler()
scaler.fit(X_seq[idx_train].reshape(-1, X_seq.shape[-1]))  # flatten (ships, months) to fit

def scale_sequences(X):
    shape = X.shape
    return scaler.transform(X.reshape(-1, shape[-1])).reshape(shape)

X_train_t = torch.tensor(scale_sequences(X_seq[idx_train]), dtype=torch.float32)
X_val_t = torch.tensor(scale_sequences(X_seq[idx_val]), dtype=torch.float32)
X_test_t = torch.tensor(scale_sequences(X_seq[idx_test]), dtype=torch.float32)

y_train_t = torch.tensor(y_seq[idx_train], dtype=torch.float32).view(-1, 1)
y_val_t = torch.tensor(y_seq[idx_val], dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_seq[idx_test], dtype=torch.float32).view(-1, 1)

X_train_t.shape

Define un pequeño pronosticador LSTM: procesa la secuencia de 11 meses, toma el **último** estado oculto (un resumen de toda la secuencia — exactamente lo que anticipó el §9 de `NB11`), y lo transforma en un único valor predicho:

In [ ]:
import torch.nn as nn

class FuelLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])

torch.manual_seed(42)
lstm_model = FuelLSTM(n_features=X_train_t.shape[-1])
sum(p.numel() for p in lstm_model.parameters())

Entrena con la misma receta que todas las redes desde `NB11`: Adam, siguiendo la pérdida de train y de validación. Como estamos prediciendo un valor continuo (consumo de combustible, no una clase), usamos **pérdida MSE** — la métrica de regresión de `NB07` — en vez de la entropía cruzada binaria de `NB11`/`NB12`/`NB13`:

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01)

n_epochs = 150
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    lstm_model.train()
    optimizer.zero_grad()
    outputs = lstm_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    lstm_model.eval()
    with torch.no_grad():
        val_loss = criterion(lstm_model(X_val_t), y_val_t)
    val_losses.append(val_loss.item())

plt.plot(train_losses, label="Training loss (MSE)")
plt.plot(val_losses, label="Validation loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM training: forecasting month-12 fuel consumption")
plt.legend()
plt.show()

**Pruébalo tú mismo**: la Sección 5 defendía que LSTM debería manejar dependencias más largas mejor que una RNN simple. Comprueba esa afirmación directamente — cambia `nn.LSTM` por `nn.RNN` (mismo patrón de constructor, pero su `forward` devuelve solo `(output, h_n)`, sin estado de celda separado) y entrena un modelo equivalente sobre exactamente los mismos datos. ¿Cuánto difieren las pérdidas de validación finales?

In [ ]:
class FuelRNN(nn.Module):
    def __init__(self, n_features, hidden_size=16):
        super().__init__()
        self.rnn = nn.RNN(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.head(h_n[-1])

torch.manual_seed(42)
rnn_model = FuelRNN(n_features=X_train_t.shape[-1])
optimizer_rnn = torch.optim.Adam(rnn_model.parameters(), lr=0.01)

rnn_val_losses = []
for epoch in range(n_epochs):
    rnn_model.train()
    optimizer_rnn.zero_grad()
    loss = criterion(rnn_model(X_train_t), y_train_t)
    loss.backward()
    optimizer_rnn.step()

    rnn_model.eval()
    with torch.no_grad():
        rnn_val_losses.append(criterion(rnn_model(X_val_t), y_val_t).item())

print(f"LSTM final validation loss: {val_losses[-1]:.1f}")
print(f"Plain RNN final validation loss: {rnn_val_losses[-1]:.1f}")


Con secuencias tan cortas (11 pasos) y tan pocos datos, la RNN simple y la LSTM suelen quedar muy cerca — la ventaja frente al desvanecimiento del gradiente para la que está construida la LSTM tiene más margen para notarse con secuencias más largas o datasets más grandes. Eso no invalida la teoría de la Sección 5; solo significa que esta comparación práctica concreta puede no ser donde la diferencia se vea más.

---

## 9. Evaluar la LSTM frente a una base de referencia honesta

Un modelo solo impresiona en relación con algo más simple. La base de referencia obvia para una serie temporal: **predecir que el mes siguiente es igual al anterior** (mes 12 ≈ `fuel_consumption` del mes 11) — sin ningún aprendizaje, solo persistencia. Si la LSTM no puede superar esto, no está justificando su complejidad:

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Naive baseline: predict month 12 = month 11's fuel_consumption
baseline_pred = X_seq[idx_test][:, -1, 1]  # last month in the input sequence, fuel_consumption column
baseline_mae = mean_absolute_error(y_seq[idx_test], baseline_pred)

# LSTM prediction
lstm_model.eval()
with torch.no_grad():
    lstm_pred = lstm_model(X_test_t).numpy().ravel()
lstm_mae = mean_absolute_error(y_seq[idx_test], lstm_pred)
lstm_r2 = r2_score(y_seq[idx_test], lstm_pred)

print(f"Naive baseline MAE: {baseline_mae:.1f} L")
print(f"LSTM MAE:           {lstm_mae:.1f} L")
print(f"LSTM R2:             {lstm_r2:.3f}")

**Lee tus propios números**: ¿supera la LSTM a la base de referencia ingenua? Con solo 72 secuencias de entrenamiento (una por buque de entrenamiento) y 11 pasos temporales cada una, este es un dataset genuinamente pequeño para una red neuronal — es perfectamente posible que gane aquí la base de referencia ingenua, y `eso sería un resultado legítimo y honesto, no un fallo del notebook`. Recuerda la lección más general del §8 de `NB11`: las redes neuronales necesitan más datos que los métodos basados en árboles para mostrar una ventaja clara, y 72 secuencias es poco incluso para los estándares de este curso. Una comprobación visual ayuda a interpretar el resultado, sea cual sea:

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_seq[idx_test], lstm_pred, alpha=0.7, label="LSTM")
plt.scatter(y_seq[idx_test], baseline_pred, alpha=0.7, label="Naive baseline", marker="x")
lims = [y_seq.min(), y_seq.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual month-12 fuel consumption (L)")
plt.ylabel("Predicted (L)")
plt.title("LSTM vs. naive baseline, test ships")
plt.legend()
plt.show()

**Pruébalo tú mismo**: desglosa el error absoluto de la LSTM por `ship_type` — ¿es la previsión del modelo igual de fiable en los cuatro tipos de buque, o le cuesta más con alguno?

In [ ]:
test_ship_ids = ship_ids[idx_test]
ship_types = fuel.drop_duplicates("ship_id").set_index("ship_id")["ship_type"]

errors_df = pd.DataFrame({
    "ship_type": [ship_types[s] for s in test_ship_ids],
    "abs_error": np.abs(y_seq[idx_test] - lstm_pred),
})
errors_df.groupby("ship_type")["abs_error"].agg(["mean", "count"])


---

## Resumen de la clase

- Los datos secuenciales llevan significado en su orden; una celda RNN reutiliza los mismos pesos en cada paso temporal, llevando un estado oculto hacia delante como memoria.
- Calculamos a mano los estados ocultos de una RNN pequeña, y vimos por qué la multiplicación repetida a lo largo de muchos pasos temporales hace que los gradientes se desvanezcan.
- El estado de celda con puertas de la LSTM, actualizado sobre todo mediante suma, resiste esto mucho mejor que una RNN simple.
- Replanteamos los datos reales de combustible de buques — ya usados dos veces en este curso como datos tabulares fila-independientes — como lo que realmente son: una serie temporal de 12 meses por buque, dividida *por buque* para evitar fugas.
- Entrenamos un pronosticador LSTM real y lo juzgamos frente a una base de referencia ingenua y honesta, no solo frente a su propia pérdida de entrenamiento.

## Para la próxima clase (NB15)

**Transfer learning**: en vez de entrenar una CNN desde cero como en `NB13`, reutilizaremos una red ya entrenada con millones de imágenes — dando continuación directa a la idea con la que cerraba `NB13` sobre por qué los primeros filtros convolucionales generalizan entre tareas.

## Tarea / Ideas de práctica

1. Cambia `hidden_size` de 16 a 32 y a 4 — ¿cómo cambia el MAE de test? Relaciona esto con la discusión sobre número de parámetros/sobreajuste de `NB11`–`NB13`.
2. Cambia `nn.LSTM` por `nn.GRU` en la Parte 8 (los mismos argumentos del constructor funcionan, aunque `nn.GRU` devuelve solo `(output, h_n)`, sin estado de celda separado) — ¿rinde de forma distinta con tan pocos datos?
3. Añade `route_id` o `weather_conditions` (codificado one-hot) como características extra de la secuencia — ¿mejora la LSTM, y merece la pena la complejidad añadida dado lo pequeño que es este dataset?
4. Prueba a predecir el mes 6 a partir de los meses 1-5 en vez del mes 12 a partir de los meses 1-11 — ¿hace una secuencia más corta que el problema del desvanecimiento del gradiente de la Parte 5 sea menos relevante en la práctica?
5. Explica con tus propias palabras por qué era necesario dividir por *buque* (Parte 7) en vez de por *fila* — ¿qué se filtraría exactamente si dividiéramos aleatoriamente por fila en su lugar?

> ***Como siempre: un modelo que pierde frente a una base de referencia de una sola línea es un resultado real y útil — te dice que los datos (o la cantidad de ellos) no soportan la complejidad que intentaste, y merece la pena saberlo antes de desplegar nada.***